# Audit — ML.ENERGY Benchmark V3

**Source :** [](https://huggingface.co/datasets/ml-energy/benchmark-v3)  
**Licence :** Apache 2.0  
**Publication :** NeurIPS Datasets & Benchmarks 2025  
**Contenu :** 46 LLMs mesurés sur GPU NVIDIA H100 et B200, ~7 tâches benchmark  

## Objectif de ce notebook

1. Charger et explorer le dataset ML.ENERGY
2. Comprendre la structure et les colonnes disponibles
3. Identifier les modèles en commun avec Compar:IA
4. Valider le mapping des noms avant insertion en BDD
5. Analyser la distribution des consommations énergétiques par tâche

**Prérequis :**

Et avoir défini  (voir  pour les instructions).

In [5]:
pip install seaborn matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\niang\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.4f}".format)

print("Packages chargés ✓")

Packages chargés ✓


## 1. Chargement du dataset

In [8]:
pip install datasets huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 555.1/555.1 kB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/693.4 kB ? eta -:--:--
   ---------------------------------------- 693.4/693.4 kB 14.1 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 19.8 MB/s  0:00:00

   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------------------------  1/20 [tqdm]
   -- -------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\niang\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
from datasets import load_dataset

# Nécessite HF_TOKEN défini en variable d'environnement
# export HF_TOKEN=hf_xxxxxxxxxxxx
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise EnvironmentError(
        "HF_TOKEN non défini. Voir instructions dans scripts/05_add_benchmark.py"
    )

ds = load_dataset(
    "ml-energy/benchmark-v3",
    name="llm",
    token=hf_token,
    split="train"
)
df = ds.to_pandas()
print(f"Dataset chargé : {len(df)} lignes, {df.shape[1]} colonnes")
print(f"Modèles : {df['model_id'].nunique()}")
print(f"Tâches  : {df['task'].nunique()} → {sorted(df['task'].unique())}")
print(f"GPU     : {sorted(df['gpu_model'].unique())}")

OSError: HF_TOKEN non défini. Voir instructions dans scripts/05_add_benchmark.py

## 2. Exploration de la structure

In [ ]:
print("=== Colonnes disponibles ===")
print(df.dtypes.to_string())
print(f"
Valeurs manquantes :")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

In [ ]:
print("=== Aperçu des données ===")
df[[
    "task", "nickname", "gpu_model", "num_gpus",
    "energy_per_token_joules", "avg_power_watts",
    "output_throughput_tokens_per_sec", "mean_itl_ms"
]].head(10)

In [ ]:
# Conversion Joules → kWh par 1000 tokens (pour comparaison avec Compar:IA)
JOULES_TO_KWH = 1 / 3_600_000
df["kwh_par_1k_tokens"] = df["energy_per_token_joules"] * 1000 * JOULES_TO_KWH

print("Statistiques energy_per_token_joules par tâche :")
df.groupby("task")["energy_per_token_joules"].describe().round(4)

## 3. Liste des modèles disponibles

In [ ]:
modeles_ml = df[["model_id", "nickname"]].drop_duplicates().sort_values("nickname")
print(f"{len(modeles_ml)} modèles ML.ENERGY :")
for _, r in modeles_ml.iterrows():
    print(f"  {r['model_id']:<55}  ({r['nickname']})")

## 4. Mapping avec les modèles Compar:IA

In [ ]:
import sqlite3

DB_PATH = "data/db/impact_ia.db"
if not os.path.exists(DB_PATH):
    print(f"BDD introuvable : {DB_PATH}")
    print("Lancer d'abord : python scripts/02_build_db.py")
else:
    conn = sqlite3.connect(DB_PATH)
    modeles_bdd = pd.read_sql("SELECT mdl_id, mdl_nom FROM Modele ORDER BY mdl_nom", conn)
    conn.close()
    print(f"{len(modeles_bdd)} modèles dans notre BDD (Compar:IA)")
    print(modeles_bdd.head(20).to_string(index=False))

In [ ]:
# Mapping des noms (même logique que 05_add_benchmark.py)
MLENERGY_TO_COMPARAIA = {
    "Llama-3.1-8B": "llama-3.1-8b",
    "Llama-3.1-70B": "llama-3.1-70b",
    "Llama-3.1-405B": "llama-3.1-405b",
    "Llama-3.3-70B": "llama-3.3-70b",
    "Llama-4-Scout": "llama-4-scout",
    "Llama-4-Maverick": "llama-maverick",
    "Mistral-Small-2506": "mistral-small-2506",
    "Mistral-Large-2512": "mistral-large-2512",
    "Mixtral-8x7B": "mixtral-8x7b-instruct-v0.1",
    "Mixtral-8x22B": "mixtral-8x22b-instruct-v0.1",
    "gemma-2-9b": "gemma-2-9b-it",
    "gemma-3-12b": "gemma-3-12b",
    "gemma-3-27b": "gemma-3-27b",
    "Qwen2.5-7B": "qwen2.5-7b-instruct",
    "Qwen2.5-32B": "qwen2.5-32b-instruct",
    "Qwen3-32B": "qwen3-32b",
    "QwQ-32B": "qwq-32b",
    "DeepSeek-V3": "deepseek-v3-chat",
    "DeepSeek-R1-0528": "deepseek-r1-0528",
}

matched = []
unmatched = []
for model_id in sorted(df["model_id"].unique()):
    short = model_id.split("/")[-1]
    found = None
    for fragment, nom_bdd in MLENERGY_TO_COMPARAIA.items():
        if fragment.lower() in short.lower():
            found = nom_bdd
            break
    if found:
        matched.append((model_id, found))
    else:
        unmatched.append(model_id)

print(f"✅ Matchés ({len(matched)}) :")
for ml, bdd in matched:
    print(f"  {ml:<55} → {bdd}")
print(f"
❌ Non matchés ({len(unmatched)}) :")
for m in unmatched:
    print(f"  {m}")
print(f"
Taux de correspondance : {len(matched)}/{len(matched)+len(unmatched)} "
      f"({100*len(matched)/(len(matched)+len(unmatched)):.0f}%)")

## 5. Analyse des consommations par tâche

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---- Boxplot par tâche ----
df_stable = df[df.get("is_stable", pd.Series([True]*len(df)))] if "is_stable" in df.columns else df
task_order = df_stable.groupby("task")["energy_per_token_joules"].median().sort_values().index

sns.boxplot(
    data=df_stable, x="energy_per_token_joules", y="task",
    order=task_order, palette="Blues_r", ax=axes[0]
)
axes[0].set_title("Distribution énergie par tâche (J/token)", fontsize=13)
axes[0].set_xlabel("Énergie (Joules / token)")
axes[0].set_ylabel("Tâche")

# ---- Top 10 modèles les plus frugaux (chat / toutes tâches) ----
df_mean = df_stable.groupby(["nickname", "gpu_model"])["energy_per_token_joules"].mean()\n
          .reset_index().sort_values("energy_per_token_joules")
df_top = df_mean[df_mean["gpu_model"] == "H100"].head(12)

bars = axes[1].barh(df_top["nickname"], df_top["energy_per_token_joules"],
                    color=sns.color_palette("Blues_r", len(df_top)))
axes[1].set_title("Top 12 modèles frugaux — GPU H100 (toutes tâches)", fontsize=13)
axes[1].set_xlabel("Énergie moyenne (J/token)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("notebooks/benchmark_energie_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure sauvegardée → notebooks/benchmark_energie_distribution.png")

## 6. Analyse H100 vs B200

In [ ]:
# Comparaison H100 vs B200 pour les modèles disponibles sur les deux GPU
pivot = df.pivot_table(
    index=["model_id", "task"],
    columns="gpu_model",
    values="energy_per_token_joules",
    aggfunc="mean"
).reset_index()

if "H100" in pivot.columns and "B200" in pivot.columns:
    pivot_valid = pivot.dropna(subset=["H100", "B200"]).copy()
    pivot_valid["gain_B200_pct"] = ((pivot_valid["H100"] - pivot_valid["B200"]) / pivot_valid["H100"] * 100).round(1)
    print("Gain énergétique B200 vs H100 (% de réduction) :")
    print(pivot_valid[["model_id", "task", "H100", "B200", "gain_B200_pct"]]
          .sort_values("gain_B200_pct", ascending=False)
          .to_string(index=False))
else:
    print("Un seul type de GPU disponible dans le dataset filtré.")

## 7. Préparation du ratio théorique/réel (Détecteur de Gaspillage)

In [ ]:
# Agrégation des mesures benchmark par modèle (moyenne toutes tâches, H100)
df_h100 = df[df["gpu_model"] == "H100"].copy() if "H100" in df["gpu_model"].values else df.copy()

benchmark_par_modele = df_h100.groupby("model_id").agg(
    nb_taches=("task", "nunique"),
    joules_moy=("energy_per_token_joules", "mean"),
    joules_med=("energy_per_token_joules", "median"),
    kwh_1k_moy=("kwh_par_1k_tokens", "mean"),
    watt_moy=("avg_power_watts", "mean"),
).reset_index().round(6)

print("Benchmark agrégé par modèle (H100) :")
print(benchmark_par_modele.to_string(index=False))

# Sauvegarde pour référence
benchmark_par_modele.to_csv("notebooks/benchmark_par_modele_H100.csv", index=False)
print("
→ Sauvegardé : notebooks/benchmark_par_modele_H100.csv")

## Conclusion

### Ce notebook a permis de :
- Vérifier la structure du dataset ML.ENERGY Benchmark V3
- Identifier les modèles en commun avec Compar:IA
- Valider le mapping des noms de modèles
- Comprendre la distribution des consommations énergétiques par tâche

### Prochaine étape :
